# Pipeline v2 Workspace

Explore the ResultsStore, inspect completed runs, and load result blobs.

In [17]:
%reload_ext autoreload
%autoreload 2
from dimensionality_manuscript.registry import RegistryPaths
from dimensionality_manuscript import ResultsStore, CVPCAConfig, SubspaceConfig, StimSpaceConfig, get_data_config
from dimensionality_manuscript.scripts.status import status

## Store overview

In [20]:
store = ResultsStore(RegistryPaths().pipeline_v2_db_path)
status()

Store: D:\localData\dimensionality-manuscript\cache\pipeline_v2\results.db
Total rows: 15262
Snapshots: 47

                              count  stored  sessions                          earliest                            latest
analysis_type schema_version                                                                                             
cvpca         v2               9153    9153       148  2026-03-17T11:11:57.090909+00:00  2026-03-17T17:24:11.514117+00:00
expmax        v1                149     149       149  2026-04-23T18:18:36.436743+00:00  2026-04-23T18:44:39.170547+00:00
locprediction v1                596     596       149  2026-05-01T07:40:57.464668+00:00  2026-05-01T09:05:50.354665+00:00
population    v1                298     298       149  2026-03-10T15:14:19.696067+00:00  2026-04-20T10:07:28.474555+00:00
regression    v1               3874    3874       149  2026-03-09T15:06:16.237990+00:00  2026-05-11T10:59:39.833010+00:00
subspace      v3               1192   

In [18]:
with store._connect() as conn:
    rows = conn.execute("SELECT * FROM results WHERE analysis_type = 'expmax'").fetchall()

print(len(rows))
for row in rows:
    print(row)
    break

149
('a9fbc1da6e6522bd', 'ATL012.2023-01-20.702', 'b5c627d5a4a1ec69', 'expmax_norm_method=zero-one_speed_threshold=1.0_num_bins=100_train_test_split=(0.8, 0.2)_smooth_width=0.25_num_steps=10_reliability_cutoff=0.1_use_huber_irls=False_huber_k=2.5_huber_eps=1e-06_uniq_val_window=501_uniq_val_interp_points=201_v1', 'expmax', 'v1', 1, b'\x80\x04\x95\x80\xbf\x00\x00\x00\x00\x00\x00}\x94(\x8c\nem_test_r2\x94G\xc6\xaa2"\xdd\t\x9f\x0b\x8c\nem_null_r2\x94G?\xbc7>\xc6\x91\xa2{\x8c\x0bem_test_rms\x94G?\xa7y\xbb\x81\x9c\xfeE\x8c\x0bem_null_rms\x94G?\xa2\x99w\xdd\xd3T\xdc\x8c\x0bem_test_mse\x94G?i\xd7\xb9un\x18s\x8c\x0bem_null_mse\x94G?a\x99\x8f9L\x91\xe5\x8c\x08step_mse\x94\x8c\x15numpy.core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K\n\x85\x94h\x0b\x8c\x05dtype\x94\x93\x94\x8c\x02f8\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01<\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK\x00t\x94b\x89CP\xdb\x84c\x18\xdc|b?\x05\xda\

In [19]:
# Invalidating old results (remember this will remove the data!!!!)
store.invalidate(schema_version="v1", analysis_type="stimspace")
# store.invalidate(schema_version="v3", analysis_type="subspace")

## Browse the summary table

In [4]:
df = store.summary_table(as_dataframe=True)
df

,result_uid,session_id,data_key,analysis_key,data_summary,analysis_summary,analysis_type,schema_version,result_stored,snapshot_path,computed_at
0,b16224640466df9d,ATL012.2023-01-20.702,ce76ae68bcb57797,21c6ec9d87ec91de,default,regression_external_placefield_1d_spks=oasis_m...,regression,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T15:06:16.237990+00:00
1,bec22bcfd16823ec,ATL012.2023-01-20.702,ce76ae68bcb57797,5f2d96cfa23efbfe,default,regression_internal_placefield_1d_spks=oasis_m...,regression,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T15:06:16.275581+00:00
2,81c8d441a3e1cbbd,ATL012.2023-01-20.702,ce76ae68bcb57797,1c6661ba37ee51a8,default,regression_external_placefield_1d_gain_spks=oa...,regression,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T15:06:16.313555+00:00
3,84c5b188833e50da,ATL012.2023-01-20.702,ce76ae68bcb57797,546733132829fe0d,default,regression_internal_placefield_1d_gain_spks=oa...,regression,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T15:06:16.344875+00:00
4,36b673b757081e44,ATL012.2023-01-20.702,ce76ae68bcb57797,946b7ba41788f51f,default,regression_external_placefield_1d_vector_gain_...,regression,v1,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-09T15:06:16.374247+00:00
...,...,...,...,...,...,...,...,...,...,...,...
12266,e2e6c6525c11ca9a,ATL076.2025-08-26.702,ce76ae68bcb57797,f024ec4809c1e115,default,subspace_covcov_crossvalidated_subspace_spks=o...,subspace,v2,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-18T00:00:08.311665+00:00
12267,0d566c2afb09d441,ATL076.2025-08-27.702,ce76ae68bcb57797,f024ec4809c1e115,default,subspace_covcov_crossvalidated_subspace_spks=o...,subspace,v2,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-18T00:01:43.023883+00:00
12268,1451a9c64a6ef681,ATL076.2025-08-27.702,ce76ae68bcb57797,6d2e67a4758b81ac,default,subspace_covcov_crossvalidated_subspace_spks=o...,subspace,v2,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-18T00:01:46.880794+00:00
12269,58b30788e8eb4d41,ATL076.2025-08-28.702,ce76ae68bcb57797,f024ec4809c1e115,default,subspace_covcov_crossvalidated_subspace_spks=o...,subspace,v2,1,D:\localData\dimensionality-manuscript\cache\p...,2026-03-18T00:02:22.026501+00:00


## Load a result by reconstructing configs from the table

Pick a row, reconstruct the `DataConfig` and `CVPCAConfig` that produced it, and retrieve the result blob.

In [ ]:
# Pick the first row as an example
analysis_type = "subspace"
row = df[df["analysis_type"] == analysis_type].iloc[0]
print(f"result_uid:       {row['result_uid']}")
print(f"session_id:       {row['session_id']}")
print(f"analysis_summary: {row['analysis_summary']}")
print(f"computed_at:      {row['computed_at']}")
print(f"has results:      {row['result_stored'] is not None}")

In [ ]:
# Reconstruct the configs that produced this row
_config = {
    "cvpca": CVPCAConfig,
    "subspace": SubspaceConfig,
}
Config = _config[analysis_type]
acfg = Config.from_key(row["analysis_key"])

print(f"AnalysisConfig: {acfg}")
result = store.get(row["session_id"], acfg)
print(result)

In [ ]:
# Load the result blob using the reconstructed config
result = store.get(row["session_id"], acfg)
print(f"Result type: {type(result)}")
print(f"Result keys: {list(result.keys())}")

# Or load directly by uid (no config reconstruction needed)
result2 = store.get_by_uid(row["result_uid"])
assert result2.keys() == result.keys()

## Inspect result contents

In [53]:
import numpy as np

for key, val in result.items():
    if isinstance(val, np.ndarray):
        print(f"  {key}: ndarray {val.shape} {val.dtype}")
    else:
        print(f"  {key}: {type(val).__name__} = {val}")

  score: dict = {'evaluation_score': 0.8219407796859741, 'variance_activity': array([4201305.5  , 2837984.2  , 2374785.8  , 2104722.5  , 1982361.8  ,
       1778306.9  , 1668157.5  , 1607227.6  , 1509142.8  , 1384007.5  ,
       1287105.5  , 1268382.5  , 1232483.6  , 1122642.2  , 1011884.7  ,
        994975.6  ,  932071.7  ,  871920.5  ,  839400.25 ,  810524.   ,
        789939.5  ,  770502.56 ,  756817.5  ,  696971.1  ,  667126.1  ,
        658578.1  ,  639118.25 ,  622496.25 ,  599270.8  ,  585717.2  ,
        567662.25 ,  556298.6  ,  530814.44 ,  521958.7  ,  504632.75 ,
        473748.5  ,  453149.84 ,  435737.6  ,  418054.84 ,  411632.66 ,
        403775.44 ,  392681.78 ,  373456.6  ,  369641.7  ,  351501.34 ,
        344326.12 ,  335710.06 ,  324626.25 ,  315128.5  ,  309177.12 ,
        299983.7  ,  296766.72 ,  289367.   ,  284285.9  ,  274875.16 ,
        266870.1  ,  260432.38 ,  256101.8  ,  243424.5  ,  240572.31 ,
        233080.9  ,  221913.86 ,  219564.89 ,  211560.89 ,

## Compare with legacy workflow results

Load the same result from the old `measure_cvpca.py` file-based cache and compare side-by-side. The trial folds are randomized independently, so results won't match exactly — but the structure and scale should agree.

In [ ]:
import joblib
from dimensionality_manuscript.registry import RegistryPaths


def get_legacy_filepath(session_id: str, acfg: CVPCAConfig):
    """Reconstruct the legacy measure_cvpca.py filepath from a CVPCAConfig."""
    from pathlib import Path

    rp = RegistryPaths()
    name = session_id
    if not acfg.center:
        name += "_notcentered"
    if acfg.use_fast_sampling:
        name += "_fast"
    if not acfg.normalize:
        name += "_nonorm"
    if acfg.reliability_threshold is not None:
        name += f"_rel{acfg.reliability_threshold:.2f}"
    if acfg.fraction_active_threshold is not None:
        name += f"_frac{acfg.fraction_active_threshold:.2f}"
    return rp.measure_cvpca_path / f"{name}.pkl"


# Pick a row to compare (change iloc index to pick a different one)
row = df[df["analysis_type"] == "cvpca"].iloc[0]
acfg = CVPCAConfig.from_key(row["analysis_key"])

# Load pipeline result
v2_result = store.get(row["session_id"], acfg)

# Load legacy result
legacy_path = get_legacy_filepath(row["session_id"], acfg)
print(f"Legacy path: {legacy_path}")
print(f"Exists: {legacy_path.exists()}")

if legacy_path.exists():
    legacy_result = joblib.load(legacy_path)
    print(f"\nv2 keys:     {sorted(v2_result.keys())}")
    print(f"legacy keys: {sorted(legacy_result.keys())}")
else:
    legacy_result = None
    print("Legacy result not found — run measure_cvpca.py first for this config.")

In [28]:
# Compare array shapes and print per-key stats
if legacy_result is not None:
    shared_keys = sorted(set(v2_result.keys()) & set(legacy_result.keys()))
    for key in shared_keys:
        v2_val = v2_result[key]
        leg_val = legacy_result[key]
        if isinstance(v2_val, np.ndarray) and isinstance(leg_val, np.ndarray):
            # Truncate to shared length (legacy may have fewer bins)
            n = min(len(v2_val), len(leg_val))
            corr = np.corrcoef(v2_val[:n].astype(float), leg_val[:n].astype(float))[0, 1]
            print(f"  {key:35s}  v2={v2_val.shape}  legacy={leg_val.shape}  corr={corr:.4f}")
        else:
            print(f"  {key:35s}  v2={type(v2_val).__name__}  legacy={type(leg_val).__name__}")

  org_covariances                      v2=(100,)  legacy=(100,)  corr=0.9984
  org_fixed_smooth_covariances         v2=(100,)  legacy=(100,)  corr=0.9986
  org_smooth_covariances               v2=(100,)  legacy=(100,)  corr=0.9991
  pca_covariances                      v2=(100,)  legacy=(100,)  corr=0.9996
  pca_fixed_smooth_covariances         v2=(100,)  legacy=(100,)  corr=0.9997
  pca_smooth_covariances               v2=(100,)  legacy=(100,)  corr=0.9999
  reg_covariances                      v2=(100,)  legacy=(100,)  corr=0.9989
  reg_covariances_fixed                v2=(100,)  legacy=(100,)  corr=0.9976
  saved_leg_covariances                v2=(80,)  legacy=(80,)  corr=1.0000
  smoothing_widths                     v2=(360,)  legacy=(360,)  corr=0.6147
  trial_folds                          v2=list  legacy=list


In [ ]:
from dimensionality_manuscript.registry import RegistryPaths
from dimensionality_manuscript import CVPCAConfig, ResultsAggregator, ResultsStore
from vrAnalysis.database import get_database

rp = RegistryPaths()
store = ResultsStore(rp.pipeline_v2_db_path)
sessions = list(get_database("vrSessions").iter_sessions(imaging=True))

In [ ]:
results = ResultsAggregator(SubspaceConfig, store, sessions)

print("param_axes:")
for name, vals in results.param_axes.items():
    print(f"  {name}: {vals}")
print()
print(f"sessions: {len(results.session_ids)}")
print()
print("array shapes:  (n_sess, *param_dims, max_result_len)")
for key, arr in results.arrays.items():
    print(f"  {key}: {arr.shape}")

In [ ]:
# Slice to center=True, normalize=True — those two dims are squeezed out
sliced, names = results.sel(return_param_sizes=True, subspace_name="covcov_subspace")
print(names)
print("After sel(center=True, normalize=True):")
for key, arr in sliced.items():
    print(f"  {key}: {arr.shape}")

['session', 'smooth_width']
After sel(center=True, normalize=True):
  variance_placefields: (149, 2, 300)
  evaluation_score: (149, 2)
  variance_activity: (149, 2, 300)
  cross: (149, 2)


In [17]:
results.param_axes.keys()

dict_keys(['subspace_name', 'spks_type', 'num_bins', 'smooth_width'])